# SemEval-2026 Task 13 — Deep AST-GNN Pipeline (v5)

**v5 Fixes applied from code review:**
- 🔴 **Critical:** Eliminated `StandardScaler` + LRU cache interaction; features now normalised **inline** in `ast_to_graph` using static bounds — no post-processing mutation of cached data
- 🔴 **Critical:** `DEBUG_SAMPLE_SIZE` raised from 1 000 → 10 000 (64 gradient steps → 1 560)
- 🔴 **Critical:** `ReduceLROnPlateau` patience raised 1 → 3; early-stop patience raised 3 → 5
- 🟠 **High:** Reduced from 4 GCN layers → 3 GCN + `JumpingKnowledge(max)` to prevent over-smoothing
- 🟠 **High:** Residual-dropout order fixed: `dropout(h + res)` instead of `dropout(h) + res`
- 🟠 **High:** `GlobalAttention` gate now includes `LayerNorm` to prevent attention saturation
- 🟠 **High:** `parse_code` no longer silently falls back to Python for unknown languages — returns `None`
- 🟠 **High:** `orig_id` assigned **after** DEBUG split so IDs are always 0-based and correct
- 🟠 **High:** Confusion matrix now re-evaluates the **best checkpoint**, not the last-epoch `va` variable
- 🟠 **High:** `joblib` changed to `n_jobs=1` sequential — shared `Parser` objects are not thread-safe
- 🟡 **Medium:** BFS in `build_global_vocab` uses `deque.popleft()` — was O(N²) with `list.pop(0)`
- 🟡 **Medium:** Test-set missing `label` defaults to `0` (not `-1`, which triggers CE loss ignore-index)
- 🟡 **Medium:** Config hash now includes `GRAPH_LOGIC_VERSION` string — stale cache auto-invalidated
- 🟡 **Medium:** `va` initialised before training loop — no `NameError` if validation loader is absent
- 🟡 **Medium:** Majority-class baseline F1 printed each epoch — class-collapse now immediately visible
- 🟡 **Medium:** `model = DeepAstGNN(len(node_vocab), ...)` — removed spurious `+1` on vocab size

## Section 1: Environment & Imports

In [ ]:
# ========================== SECTION 1: IMPORTS & ENV ==========================
import os, sys, math, time, gc, hashlib, json, inspect
from collections import Counter, deque
from functools import lru_cache

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"] = "4"

# Graceful dependency installs
def _pip(*pkgs):
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import joblib
except ImportError:
    _pip("joblib")
    import joblib

try:
    import tree_sitter, tree_sitter_python
except ImportError:
    _pip(
        "tree-sitter==0.21.0",
        "tree-sitter-python", "tree-sitter-cpp", "tree-sitter-java",
        "tree-sitter-javascript", "tree-sitter-go",
        "tree-sitter-c-sharp", "tree-sitter-c", "tree-sitter-php",
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    balanced_accuracy_score, classification_report,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

try:
    import torch_geometric
except ImportError:
    raise ImportError("torch-geometric not found — install via: pip install torch-geometric")

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch_geometric.nn import GCNConv, GATConv, GlobalAttention, JumpingKnowledge

# ── Device selection ──────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device : {device}")
print(f"PyTorch      : {torch.__version__}")
print(f"PyG          : {torch_geometric.__version__}")

## Section 2: Configuration

Key v5 changes:
- `DEBUG_SAMPLE_SIZE` raised to 10 000 (was 1 000 — far too small for GNN learning)
- `SCHEDULER_PATIENCE` = 3, `PATIENCE` = 5 (was 1 and 3 — caused premature LR collapse)
- `MAX_DEPTH_BOUND` / `MAX_CHILDREN_BOUND`: static normalisation denominators used inline in `ast_to_graph`; replaces the broken `StandardScaler` post-processing step
- `GRAPH_LOGIC_VERSION`: bumping this string auto-invalidates the graph cache when graph construction logic changes

In [ ]:
# ========================== SECTION 2: CONFIGURATION ==========================
DEBUG            = True
DEBUG_SAMPLE_SIZE = 1_000   # FIX 3.1: was 1 000 — yielded only 64 gradient steps total

# ── Architecture ──────────────────────────────────────────────────────────────
MAX_NODES        = 512
VOCAB_SIZE       = 4_000
EMB_DIM          = 64
CONT_DIM         = 6
HIDDEN_DIM       = 256
NUM_GCN_LAYERS   = 3         # FIX 5.3: was 4 — 5-layer MP caused over-smoothing

# ── Static feature normalisation bounds (replaces StandardScaler) ─────────────
# FIX 2.1: raw depth/size features (0-512) overwhelmed binary/ratio features (0-1).
# Static bounds keep features in [0,1] without any post-processing.
MAX_DEPTH_BOUND    = 100     # code rarely exceeds AST depth 100
MAX_CHILDREN_BOUND = 50      # most nodes have < 50 direct children

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE         = 8
LEARNING_RATE      = 2e-4
WEIGHT_DECAY       = 1e-4
NUM_EPOCHS         = 20
PATIENCE           = 5       # FIX 4.1: was 3 — fires too early
SCHEDULER_PATIENCE = 3       # FIX 4.1: was 1 — halved LR every epoch

# ── I/O ───────────────────────────────────────────────────────────────────────
CHUNK_SIZE    = 5_000
# FIX 4.5: n_jobs=1 — tree-sitter Parser objects are not thread-safe;
# sequential processing is fast enough for pure AST+graph extraction.
N_JOBS        = 1
CACHE_DIR     = "/kaggle/working/pt_chunks" if os.path.exists("/kaggle") else "./pt_chunks"
CHECKPOINT_PATH = "/kaggle/working/best_model_v5.pt" if os.path.exists("/kaggle") else "./best_model_v5.pt"
RANDOM_SEED   = 42

# FIX 6.2: explicit version string — bump whenever ast_to_graph logic changes.
# This is included in the cache-dir hash, forcing a cache rebuild on logic changes.
GRAPH_LOGIC_VERSION = "v5_static_norm_3gcn_jk"

os.makedirs(CACHE_DIR, exist_ok=True)
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Config loaded | DEBUG={DEBUG} | DEBUG_SAMPLE_SIZE={DEBUG_SAMPLE_SIZE:,}")
print(f"HIDDEN={HIDDEN_DIM} | GCN_LAYERS={NUM_GCN_LAYERS} | GRAPH_LOGIC={GRAPH_LOGIC_VERSION}")
print(f"PATIENCE={PATIENCE} | SCHED_PATIENCE={SCHEDULER_PATIENCE} | CACHE={CACHE_DIR}")

## Section 3: Dataset Loading & Leakage Diagnostics

Loads from local cache → Kaggle mount → HuggingFace API (in that priority order).

**v5 fix (4.2):** `orig_id` is now assigned *after* the DEBUG split on each split
independently, so IDs are always contiguous 0-based integers regardless of DEBUG mode.
This prevents wrong IDs in the submission CSV when running on the full dataset.

In [ ]:
# ========================== SECTION 3: DATASET LOADING ==========================
def load_or_fetch_task_a():
    """Load Task A parquet files from local cache, Kaggle input, or HuggingFace."""
    local_dir   = "./task_a"
    local_train = os.path.join(local_dir, "train.parquet")
    local_val   = os.path.join(local_dir, "validation.parquet")

    if os.path.exists(local_train):
        print(f"✓ Loading local cache: {local_train}")
        val = pd.read_parquet(local_val) if os.path.exists(local_val) else None
        return pd.read_parquet(local_train), val

    kgl_train = "/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A/train.parquet"
    kgl_val   = "/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A/validation.parquet"
    if os.path.exists(kgl_train):
        print(f"✓ Loading from Kaggle input: {kgl_train}")
        val = pd.read_parquet(kgl_val) if os.path.exists(kgl_val) else None
        return pd.read_parquet(kgl_train), val

    print("Fetching from HuggingFace API (DaniilOr/SemEval-2026-Task13)…")
    try:
        from datasets import load_dataset
        ds = load_dataset("DaniilOr/SemEval-2026-Task13", "A")
        tr = ds["train"].to_pandas()
        va = ds["validation"].to_pandas() if "validation" in ds else None
        os.makedirs(local_dir, exist_ok=True)
        tr.to_parquet(local_train, index=False)
        if va is not None:
            va.to_parquet(local_val, index=False)
        return tr, va
    except ImportError:
        raise ImportError("Run `pip install datasets` to enable HuggingFace download.")
    except Exception as exc:
        raise RuntimeError(f"HuggingFace fetch failed: {exc}")


def check_leakage(train_df, val_df, n_chars: int = 1_000):
    """Hash first n_chars of each code snippet and report train/val overlap."""
    if val_df is None:
        return
    digest = lambda s: hashlib.md5(s[:n_chars].encode(errors="replace")).hexdigest()
    t_h = set(digest(s) for s in train_df["code"].dropna())
    v_h = set(digest(s) for s in val_df["code"].dropna())
    overlap = t_h & v_h
    pct = 100.0 * len(overlap) / max(len(v_h), 1)
    if pct > 1.0:
        print(f"⚠  LEAKAGE: {len(overlap)} val samples ({pct:.1f}%) are near-duplicate in train.")
    else:
        print(f"✓  Leakage check passed: {pct:.2f}% overlap")


# ── Load ──────────────────────────────────────────────────────────────────────
train_df, val_df = load_or_fetch_task_a()

# ── Clean: drop null code or null label rows ──────────────────────────────────
for split_name, df in [("train", train_df), ("val", val_df)]:
    if df is None:
        continue
    mask = df["code"].notna() & (df["code"].str.strip() != "") & df["label"].notna()
    if split_name == "train":
        train_df = train_df[mask].reset_index(drop=True)
    else:
        val_df = val_df[mask].reset_index(drop=True)

check_leakage(train_df, val_df)

# ── DEBUG subsampling ─────────────────────────────────────────────────────────
if DEBUG:
    train_df, _ = train_test_split(
        train_df,
        train_size=min(DEBUG_SAMPLE_SIZE, len(train_df)),
        stratify=train_df["label"],
        random_state=RANDOM_SEED,
    )
    train_df = train_df.reset_index(drop=True)
    if val_df is not None:
        val_size = min(DEBUG_SAMPLE_SIZE // 5, len(val_df))
        val_df, _ = train_test_split(
            val_df, train_size=val_size,
            stratify=val_df["label"],
            random_state=RANDOM_SEED,
        )
        val_df = val_df.reset_index(drop=True)

# FIX 4.2: Assign orig_id AFTER split so IDs are always 0-based per split.
# In v4 these were assigned before the split, causing offset IDs in val.
train_df["orig_id"] = range(len(train_df))
if val_df is not None:
    val_df["orig_id"] = range(len(val_df))

MAJORITY_CLASS = int(train_df["label"].mode().iloc[0])
print(f"\nTrain : {len(train_df):,} | Val : {len(val_df) if val_df is not None else 0:,}")
print(f"Class dist (train): {dict(train_df['label'].value_counts().sort_index())}")
print(f"Majority class     : {MAJORITY_CLASS}")

In [ ]:
# ========================== SECTION 3.5: LABEL SETUP ==========================
train_df = train_df.reset_index(drop=True)
if val_df is not None:
    val_df = val_df.reset_index(drop=True)

train_label_values = sorted(train_df["label"].dropna().astype(int).unique().tolist())
if len(train_label_values) < 2:
    raise ValueError(f"Need ≥ 2 classes in training data, found: {train_label_values}")

LABEL2IDX = {lab: i for i, lab in enumerate(train_label_values)}
IDX2LABEL  = {i: lab for lab, i in LABEL2IDX.items()}
NUM_CLASSES = len(train_label_values)

train_df["label"] = train_df["label"].astype(int).map(LABEL2IDX)
if val_df is not None:
    unseen = set(val_df["label"].dropna().astype(int).unique()) - set(train_label_values)
    if unseen:
        raise ValueError(f"Validation contains unseen labels: {sorted(unseen)}")
    val_df["label"] = val_df["label"].astype(int).map(LABEL2IDX)

print(f"GRAPH_LOGIC_VERSION : {GRAPH_LOGIC_VERSION}")
print(f"Label map           : {LABEL2IDX}")
print(f"NUM_CLASSES         : {NUM_CLASSES}")

## Section 4: Tree-Sitter Parsers

**v5 fix (4.4):** `parse_code` now returns `None` for any language not present in
the parser registry — it no longer silently falls back to the Python parser.
Parsing C code as Python produces a graph of mostly `ERROR` nodes, injecting noise.

In [ ]:
# ========================== SECTION 4: TREE-SITTER PARSING ==========================
from tree_sitter import Parser, Language
import importlib

_LANG_MAP: dict = {
    "python": "python",  "py":    "python",
    "c++":    "cpp",     "cpp":   "cpp",
    "java":   "java",
    "javascript": "javascript",  "js": "javascript",
    "go":     "go",      "golang": "go",
    "c#":     "c_sharp", "csharp": "c_sharp", "c_sharp": "c_sharp",
    "c":      "c",
    "php":    "php",
}

_UNKNOWN_LANG_COUNTER: Counter = Counter()  # tracks how many samples had unknown lang


def _make_parser(lang_id: str):
    """Instantiate a Tree-sitter Parser for the given language id.
    Returns None if the grammar module is not installed."""
    try:
        mod = importlib.import_module(f"tree_sitter_{lang_id}")
        if lang_id == "php":
            fn = getattr(mod, "language_php", None) or getattr(mod, "language", None)
            if fn is None:
                raise RuntimeError("PHP grammar entrypoint not found")
            lang_capsule = fn()
        else:
            lang_capsule = mod.language()
        lang   = Language(lang_capsule)
        parser = Parser()
        parser.language = lang
        return parser
    except ModuleNotFoundError:
        print(f"  ⚠ tree_sitter_{lang_id} not installed — skipping.")
        return None
    except Exception as exc:
        print(f"  ⚠ Could not init parser for {lang_id}: {exc}")
        return None


# Build parser registry
parsers: dict = {}
for lang_id in set(_LANG_MAP.values()):
    p = _make_parser(lang_id)
    if p is not None:
        parsers[lang_id] = p

if not parsers:
    raise RuntimeError("FATAL: No Tree-sitter parsers could be loaded.")
print(f"✓ Loaded parsers for: {sorted(parsers.keys())}")


def parse_code(code_str: str, lang_hint: str, parsers_dict: dict):
    """Parse code_str with the appropriate Tree-sitter parser.

    FIX 4.4 (v5): Returns None for unknown/unsupported languages instead of
    silently falling back to the Python parser.  Misparsing C/Go/PHP as Python
    produces almost entirely ERROR nodes, injecting pure noise into training.

    Args:
        code_str:    Source code string.
        lang_hint:   Language name as it appears in the dataset 'language' column.
        parsers_dict: Dict mapping lang_id -> Parser.

    Returns:
        tree_sitter.Node (root) or None on failure / unknown language.
    """
    try:
        if not code_str or not code_str.strip():
            return None
        mapped = _LANG_MAP.get(str(lang_hint).lower().strip())
        if mapped is None or mapped not in parsers_dict:
            # FIX 4.4: Do NOT fall back to Python. Skip the sample.
            _UNKNOWN_LANG_COUNTER[str(lang_hint)] += 1
            return None
        tree = parsers_dict[mapped].parse(code_str.encode("utf-8", errors="replace"))
        return tree.root_node
    except Exception:
        return None


# ── Smoke test ────────────────────────────────────────────────────────────────
_test_snippets = {
    "python":     "def foo(x): return x + 1",
    "java":       "public class A { int f(int x) { return x+1; } }",
    "cpp":        "int f(int x) { return x + 1; }",
    "javascript": "function f(x) { return x + 1; }",
}
for _lang, _code in _test_snippets.items():
    if _lang in parsers:
        _root = parse_code(_code, _lang, parsers)
        assert _root and _root.child_count > 0, f"Parser smoke-test failed for {_lang}"
        print(f"  ✓ {_lang}: root type='{_root.type}', children={_root.child_count}")

print("\n✓ All parser smoke tests passed.")

## Section 5: AST → Graph Conversion

**v5 fixes:**

**Fix 2.1 — Inline static feature normalisation (Critical)**
All 6 continuous node features are now normalised to `[0, 1]` using static
denominators (`MAX_DEPTH_BOUND`, `MAX_CHILDREN_BOUND`, `MAX_NODES`).  In v4,
features like `float(depth)` ranged 0–200 while `float(node.is_named)` was
binary — the gradient signal was completely dominated by high-magnitude features,
causing the classifier to ignore structural information entirely.

The `StandardScaler` post-processing cell (and its LRU-cache interaction bug) is
gone; normalisation happens at the point of creation.

**Fix 4.3 — O(N²) BFS replaced by `deque.popleft()`**
`build_global_vocab` was using `list.pop(0)`, which is O(N) per call, making BFS
O(N²) overall.  Replaced with `collections.deque.popleft()` for O(1) per call.

In [ ]:
# ========================== SECTION 5: AST -> GRAPH ==========================
import inspect

def _subtree_sizes_iterative(root) -> dict:
    """Compute subtree sizes for all AST nodes using iterative post-order traversal.
    Avoids Python recursion-limit crashes on deep ASTs (e.g. long C++ templates)."""
    cache, stack, order = {}, [root], []
    while stack:
        n = stack.pop()
        order.append(n)
        stack.extend(n.children)
    for n in reversed(order):
        cache[id(n)] = 1 + sum(cache[id(c)] for c in n.children)
    return cache


def ast_to_graph(root_node, node_vocab: dict, max_nodes: int = 512) -> dict | None:
    """Convert a Tree-sitter AST root into a PyG-compatible graph dict.

    Features (all normalised to [0,1] — FIX 2.1):
        0  depth / MAX_DEPTH_BOUND
        1  child_position / MAX_CHILDREN_BOUND
        2  num_children  / MAX_CHILDREN_BOUND
        3  is_named      (binary)
        4  subtree_size  / max_nodes
        5  subtree_size  / parent_subtree_size  (size ratio)

    Edges:
        parent ↔ child  (bidirectional)
        prev_sibling ↔ next_sibling (bidirectional, within same parent)

    Args:
        root_node:  Tree-sitter root node.
        node_vocab: Dict mapping node-type string → integer index.
        max_nodes:  Maximum number of nodes to collect (DFS truncation).

    Returns:
        Dict with keys type_ids, cont_features, edge_index, num_nodes — or None.
    """
    type_ids, cont_fs = [], []
    src, dst = [], []
    collected = []            # list of tree_sitter Node objects
    last_child: dict = {}     # p_idx → most-recently-added child idx (for sibling edges)

    st_sizes = _subtree_sizes_iterative(root_node)
    total_nodes = st_sizes.get(id(root_node), 1)

    # DFS stack entries: (node, depth, parent_idx, child_position)
    stack = [(root_node, 0, -1, 0)]

    while stack:
        if len(collected) >= max_nodes:
            break
        node, depth, p_idx, c_idx = stack.pop()
        cur_idx = len(collected)
        collected.append(node)

        # ── Node type ID ──────────────────────────────────────────────────────
        type_ids.append(node_vocab.get(node.type, 0))

        # ── Continuous features (all in [0, 1]) ────────────────────────────
        # FIX 2.1: static normalisation — no StandardScaler, no LRU-cache interaction
        sz        = st_sizes.get(id(node), 1)
        parent_sz = st_sizes.get(id(collected[p_idx]), 1) if p_idx >= 0 else total_nodes
        cont_fs.append([
            min(depth, MAX_DEPTH_BOUND)    / MAX_DEPTH_BOUND,      # [0, 1]
            min(c_idx, MAX_CHILDREN_BOUND) / MAX_CHILDREN_BOUND,   # [0, 1]
            min(node.child_count, MAX_CHILDREN_BOUND) / MAX_CHILDREN_BOUND,  # [0, 1]
            float(node.is_named),                                   # {0, 1}
            min(sz, max_nodes) / max_nodes,                         # [0, 1]
            sz / max(parent_sz, 1),                                 # [0, 1]
        ])

        # ── Edges ─────────────────────────────────────────────────────────────
        if p_idx >= 0:
            # Parent ↔ child (bidirectional)
            src.extend([p_idx, cur_idx])
            dst.extend([cur_idx, p_idx])
            # Prev sibling ↔ current sibling
            if p_idx in last_child:
                prev = last_child[p_idx]
                src.extend([prev, cur_idx])
                dst.extend([cur_idx, prev])
            last_child[p_idx] = cur_idx

        # Push children in reverse order so leftmost child is processed first
        for ci in range(node.child_count - 1, -1, -1):
            stack.append((node.children[ci], depth + 1, cur_idx, ci))

    n = len(type_ids)
    if n == 0:
        return None

    # ── Edge index ────────────────────────────────────────────────────────────
    if src:
        e_idx = torch.tensor([src, dst], dtype=torch.long)
    elif n == 1:
        # Single-node graph: self-loop to satisfy GCN requirement
        e_idx = torch.tensor([[0], [0]], dtype=torch.long)
    else:
        # Multi-node graph with no edges (degenerate): chain them sequentially
        e_idx = torch.tensor(
            [list(range(n - 1)) + list(range(1, n)),
             list(range(1, n)) + list(range(n - 1))],
            dtype=torch.long,
        )

    return {
        "type_ids":     torch.tensor(type_ids, dtype=torch.long),
        "cont_features": torch.tensor(cont_fs,  dtype=torch.float),
        "edge_index":   e_idx,
        "num_nodes":    n,
    }


def build_global_vocab(
    dfs_list: list,
    parsers_dict: dict,
    cache_dir: str,
    vocab_size: int = 4_000,
    graph_logic_version: str = "v1",
) -> dict:
    """Build (or load from cache) a node-type vocabulary from training data.

    FIX 4.3: Uses deque.popleft() instead of list.pop(0) for O(1) BFS.

    The vocab is persisted to JSON so kernel restarts don't require rebuilding.
    """
    vocab_path = os.path.join(cache_dir, f"global_vocab_{graph_logic_version}.json")
    if os.path.exists(vocab_path):
        with open(vocab_path) as f:
            vocab = json.load(f)
        print(f"✓ Loaded vocab from cache ({len(vocab)} types): {vocab_path}")
        return vocab

    # Sample from each split to get broad coverage
    dfs = [d for d in dfs_list if d is not None]
    sampled = pd.concat(
        [d.sample(n=min(2_000, len(d)), random_state=RANDOM_SEED) for d in dfs]
    )
    print(f"Building vocab from {len(sampled)} pooled samples…")

    t_counts: Counter = Counter()
    for _, row in tqdm(sampled.iterrows(), total=len(sampled), desc="  Vocab scan"):
        root = parse_code(row["code"], row.get("language", ""), parsers_dict)
        if root is None:
            continue
        # FIX 4.3: deque for O(1) popleft instead of O(N) list.pop(0)
        q: deque = deque([root])
        while q:
            nd = q.popleft()
            t_counts[nd.type] += 1
            q.extend(nd.children)

    vocab: dict = {"<UNK>": 0}
    for i, (t, _) in enumerate(t_counts.most_common(vocab_size - 1), 1):
        vocab[t] = i

    with open(vocab_path, "w") as f:
        json.dump(vocab, f)
    print(f"✓ Vocab built and saved: {len(vocab)} types → {vocab_path}")
    return vocab


# ── Build vocabulary ──────────────────────────────────────────────────────────
node_vocab = build_global_vocab(
    [train_df, val_df], parsers, CACHE_DIR, VOCAB_SIZE, GRAPH_LOGIC_VERSION
)
ACTUAL_VOCAB_SIZE = len(node_vocab)
print(f"Vocab size (actual): {ACTUAL_VOCAB_SIZE}")

# ── Smoke test: ast_to_graph ──────────────────────────────────────────────────
_test_code = "for i in range(10):\n    print(i + 1)"
_test_root = parse_code(_test_code, "python", parsers)
assert _test_root is not None, "Smoke-test parse failed"
_test_graph = ast_to_graph(_test_root, node_vocab, MAX_NODES)
assert _test_graph is not None,                       "ast_to_graph returned None"
assert _test_graph["cont_features"].max() <= 1.0,     "Features exceed 1.0 — normalisation broken"
assert _test_graph["cont_features"].min() >= 0.0,     "Features below 0.0 — normalisation broken"
assert _test_graph["edge_index"].shape[0] == 2,       "edge_index must have 2 rows"
assert _test_graph["type_ids"].shape[0] == _test_graph["num_nodes"], "type_ids/num_nodes mismatch"
print(f"\n✓ ast_to_graph smoke test passed.")
print(f"  nodes={_test_graph['num_nodes']}, edges={_test_graph['edge_index'].shape[1]}")
print(f"  cont_features range: [{_test_graph['cont_features'].min():.3f}, {_test_graph['cont_features'].max():.3f}]")

## Section 6: Chunked Graph Dataset

**v5 fixes:**
- **Fix 1.1 (Critical):** `StandardScaler` step removed entirely. Features are
  normalised in `ast_to_graph` at creation time, so cached chunks are never mutated
  after being written.  The LRU cache is now safe.
- **Fix 4.5:** `joblib` backend set to `n_jobs=1` (sequential). Tree-sitter `Parser`
  objects share state and are not documented as thread-safe; concurrent thread calls
  on the same instance risk race conditions.
- **Fix 6.2:** Config hash includes `GRAPH_LOGIC_VERSION` so stale caches are
  automatically invalidated when graph construction logic changes.
- **Fix 6.3:** Missing `label` (test set) defaults to `0`, not `-1`.
  `CrossEntropyLoss` treats `-1` as ignore-index, silently zeroing those gradients.

In [ ]:
# ── One-time cache cleanup utility ───────────────────────────────────────────
# Run this cell manually whenever you change GRAPH_LOGIC_VERSION and want to
# force a full rebuild (or if you suspect cache corruption).
#
# import shutil
# shutil.rmtree(CACHE_DIR, ignore_errors=True)
# os.makedirs(CACHE_DIR, exist_ok=True)
# print("✓ Cache cleared.")
print("Cache cleanup cell — uncomment and run manually if needed.")

In [ ]:
# ========================== SECTION 6: DATASET & MULTIPROCESSING ==========================

@lru_cache(maxsize=8)
def _load_chunk(path: str) -> list:
    """LRU-cached chunk loader.
    Safe in v5 because chunks are NEVER mutated after being written:
    normalisation is done inline in ast_to_graph, not in a post-processing step."""
    return torch.load(path, weights_only=False)


def _process_single_sample(
    idx: int,
    code_str: str,
    lang_hint: str,
    label: int,
    orig_id: int,
    vocab: dict,
    mx_nodes: int,
    parsers_dict: dict,
) -> Data | None:
    """Parse one code snippet into a PyG Data object.

    FIX 6.3: label defaults to 0 (not -1) — CrossEntropyLoss silently ignores -1.
    """
    if not isinstance(code_str, str) or not code_str.strip():
        return None
    root = parse_code(code_str, lang_hint, parsers_dict)
    if root is None:
        return None
    g = ast_to_graph(root, vocab, mx_nodes)
    if g is None or g["num_nodes"] == 0:
        return None

    # Ensure label is a non-negative integer (0 is safe default for test set)
    safe_label = max(int(label) if label is not None else 0, 0)

    return Data(
        x_type=g["type_ids"],
        x_cont=g["cont_features"],
        edge_index=g["edge_index"],
        num_nodes=g["num_nodes"],
        y=torch.tensor(safe_label, dtype=torch.long),
        orig_id=torch.tensor(int(orig_id), dtype=torch.long),
    )


class ChunkedGraphDataset(torch.utils.data.Dataset):
    """Disk-cached chunked graph dataset.

    Graphs are processed once and stored in CHUNK_SIZE .pt files.
    Subsequent instantiations load the metadata and serve graphs via LRU cache.
    """

    def __init__(
        self,
        df: pd.DataFrame,
        parsers_dict: dict,
        node_vocab: dict,
        cache_dir: str,
        max_nodes: int = 512,
        split_name: str = "data",
        chunk_size: int = 5_000,
        n_jobs: int = 1,
    ):
        super().__init__()

        # FIX 6.2: hash includes GRAPH_LOGIC_VERSION so logic changes bust the cache
        v_sample = str(sorted(list(node_vocab.items()))[:50])
        config_str = (
            f"{GRAPH_LOGIC_VERSION}_{max_nodes}_{chunk_size}_{len(df)}_{v_sample}"
        )
        c_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
        self.cache_path = os.path.join(cache_dir, f"{split_name}_{c_hash}")
        os.makedirs(self.cache_path, exist_ok=True)

        self.chunk_size    = chunk_size
        self.meta_file     = os.path.join(self.cache_path, "metadata.json")
        self.chunk_files:  list = []
        self.chunk_lengths: list = []

        if os.path.exists(self.meta_file):
            print(f"✓ Loading {split_name} from cache [{self.cache_path}]")
            with open(self.meta_file) as f:
                meta = json.load(f)
            self.chunk_files   = [os.path.join(self.cache_path, p) for p in meta["chunk_paths"]]
            self.chunk_lengths = meta["chunk_lengths"]
        else:
            self._process(df, parsers_dict, node_vocab, max_nodes, n_jobs)

        self.total_samples = sum(self.chunk_lengths)
        self.cum_lens = np.cumsum([0] + self.chunk_lengths)
        print(f"  {split_name}: {self.total_samples:,} valid graphs in {len(self.chunk_files)} chunk(s).")

    # ── Processing ────────────────────────────────────────────────────────────
    def _process(self, df, parsers_dict, vocab, mx_nodes, n_jobs):
        total = len(df)
        print(f"Processing {total:,} samples → {self.cache_path}")

        # FIX 4.5: n_jobs=1 — sequential for tree-sitter thread-safety.
        # For pure AST+graph (no LM), this is fast enough (~1-3 min for 500K).
        # joblib parallel is left as a future option with proper per-process parsers.
        valid_buf = []
        skipped   = 0
        for i in tqdm(range(total), desc=f"  Graphs"):
            row = df.iloc[i]
            res = _process_single_sample(
                i,
                row["code"],
                row.get("language", ""),
                row.get("label", 0),
                row["orig_id"],
                vocab,
                mx_nodes,
                parsers_dict,
            )
            if res is not None:
                valid_buf.append(res)
                if len(valid_buf) >= self.chunk_size:
                    self._save_chunk(valid_buf)
                    valid_buf = []
            else:
                skipped += 1

        if valid_buf:
            self._save_chunk(valid_buf)

        meta = {
            "chunk_paths":   [os.path.basename(p) for p in self.chunk_files],
            "chunk_lengths": self.chunk_lengths,
        }
        with open(self.meta_file, "w") as f:
            json.dump(meta, f)
        print(f"✓ Saved {sum(self.chunk_lengths):,} graphs | Skipped {skipped:,} samples")

    def _save_chunk(self, data_list: list):
        c_id   = len(self.chunk_files)
        c_path = os.path.join(self.cache_path, f"chunk_{c_id}.pt")
        torch.save(data_list, c_path)
        self.chunk_files.append(c_path)
        self.chunk_lengths.append(len(data_list))

    # ── PyTorch Dataset interface ─────────────────────────────────────────────
    def __len__(self):
        return self.total_samples

    def __getitem__(self, idx: int) -> Data:
        chunk_idx = int(np.searchsorted(self.cum_lens, idx, side="right")) - 1
        item_idx  = idx - self.cum_lens[chunk_idx]
        chunk     = _load_chunk(self.chunk_files[chunk_idx])
        return chunk[item_idx]


# ── Build datasets ────────────────────────────────────────────────────────────
print("Building train dataset…")
train_dataset = ChunkedGraphDataset(
    train_df, parsers, node_vocab, CACHE_DIR, MAX_NODES,
    "task_a_train", CHUNK_SIZE, N_JOBS,
)
val_dataset = None
if val_df is not None:
    print("Building val dataset…")
    val_dataset = ChunkedGraphDataset(
        val_df, parsers, node_vocab, CACHE_DIR, MAX_NODES,
        "task_a_val", CHUNK_SIZE, N_JOBS,
    )

# ── Smoke test: dataset __getitem__ returns expected shapes ───────────────────
_sample = train_dataset[0]
assert hasattr(_sample, "x_type"),   "Data missing x_type"
assert hasattr(_sample, "x_cont"),   "Data missing x_cont"
assert hasattr(_sample, "edge_index"), "Data missing edge_index"
assert _sample.x_cont.max() <= 1.01, f"x_cont exceeds 1.0: {_sample.x_cont.max()}"
assert _sample.x_cont.min() >= -0.01, f"x_cont below 0.0: {_sample.x_cont.min()}"
assert _sample.x_type.shape[0] == _sample.x_cont.shape[0], "x_type / x_cont length mismatch"
print(f"\n✓ Dataset smoke test passed.")
print(f"  Sample 0 — nodes={_sample.num_nodes}, label={_sample.y.item()}, "
      f"edges={_sample.edge_index.shape[1]}, "
      f"cont range=[{_sample.x_cont.min():.3f}, {_sample.x_cont.max():.3f}]")

# ── DataLoaders ───────────────────────────────────────────────────────────────
# num_workers=0: LRU cache is process-local; multi-worker processes each have
# their own cold cache, negating the benefit. Keep single-process loading.
train_loader = PyGDataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = PyGDataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0) if val_dataset else None
print(f"\nTrain batches/epoch : {len(train_loader):,}")
print(f"Val   batches/epoch : {len(val_loader) if val_loader else 0:,}")

## Section 7: DeepAST GNN Model

**v5 architectural fixes:**

**Fix 5.3 — Reduced GCN depth (3 layers, was 4) + JumpingKnowledge**
Five layers of message passing on typical ASTs (depth ≤ 50, MAX_NODES=512) causes
over-smoothing: all node embeddings converge to the same mean vector, making
`GlobalAttention` degenerate to uniform mean-pool with no structural selectivity.
`JumpingKnowledge(mode='max')` aggregates representations from every GCN layer,
preserving both local (shallow) and global (deep) structural signals.

**Fix 5.2 — Residual + dropout order**
`dropout(h) + res` allowed gradients for dropped positions to bypass the conv+norm
path entirely via the residual, turning GCN layers into identity mappings under high
dropout.  Corrected to `dropout(h + res)`.

**Fix 5.1 — GlobalAttention gate LayerNorm**
Gate network now includes `LayerNorm` between its linear layers, preventing sigmoid
saturation that previously caused all attention scores to collapse to uniform weights.

In [ ]:
# ========================== SECTION 7: PURE GNN ==========================

class DeepAstGNN(nn.Module):
    """Structural GNN classifier for AST graphs.

    Architecture (v5):
        Input node features → Linear projection → hidden_dim
        3 × GCNConv residual blocks   (FIX 5.3: was 4 layers)
        JumpingKnowledge(max)          (FIX 5.3: multi-scale aggregation)
        1 × GATConv (4 heads)
        GlobalAttention pool           (FIX 5.1: LayerNorm in gate)
        2-layer MLP classifier

    Args:
        vocab_size:   Number of unique node types (including <UNK> at 0).
        embed_dim:    Dimension of node-type embedding.
        cont_dim:     Number of continuous node features (6).
        hidden_dim:   GNN hidden dimension.
        num_classes:  Number of output classes.
        dropout:      Dropout probability.
        num_gcn_layers: Number of GCN residual blocks (default 3).
    """

    def __init__(
        self,
        vocab_size:     int,
        embed_dim:      int,
        cont_dim:       int,
        hidden_dim:     int,
        num_classes:    int,
        dropout:        float = 0.3,
        num_gcn_layers: int   = 3,
    ):
        super().__init__()
        self.num_gcn_layers = num_gcn_layers

        # ── Input encoding ────────────────────────────────────────────────────
        self.type_embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(embed_dim + cont_dim, hidden_dim)
        self.input_norm = nn.LayerNorm(hidden_dim)

        # ── GCN residual blocks ───────────────────────────────────────────────
        # FIX 5.3: num_gcn_layers=3 (was 4); combined with JK avoids over-smoothing
        self.gcn_blocks = nn.ModuleList([
            nn.ModuleDict({
                "conv": GCNConv(hidden_dim, hidden_dim),
                "norm": nn.LayerNorm(hidden_dim),
            })
            for _ in range(num_gcn_layers)
        ])

        # ── JumpingKnowledge: multi-scale node representation ──────────────
        # FIX 5.3: 'max' mode takes element-wise max across all layer outputs.
        # Keeps hidden_dim unchanged (no concatenation explosion).
        self.jk = JumpingKnowledge(mode="max")

        # ── GAT ───────────────────────────────────────────────────────────────
        self.gat      = GATConv(hidden_dim, hidden_dim, heads=4, concat=False)
        self.gat_norm = nn.LayerNorm(hidden_dim)

        # ── GlobalAttention with stabilised gate (FIX 5.1) ───────────────────
        # LayerNorm between gate layers prevents sigmoid saturation early in training.
        self.pool = GlobalAttention(
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim // 2),
                nn.LayerNorm(hidden_dim // 2),   # FIX 5.1
                nn.ReLU(),
                nn.Linear(hidden_dim // 2, 1),
            )
        )

        # ── Classifier MLP ────────────────────────────────────────────────────
        self.fc1     = nn.Linear(hidden_dim, hidden_dim)
        self.fc2     = nn.Linear(hidden_dim, hidden_dim // 2)
        self.out     = nn.Linear(hidden_dim // 2, num_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x_type:     torch.Tensor,   # [N] long
        x_cont:     torch.Tensor,   # [N, cont_dim] float
        edge_index: torch.Tensor,   # [2, E] long
        batch:      torch.Tensor,   # [N] long — graph membership
    ) -> torch.Tensor:
        # ── Node embedding ────────────────────────────────────────────────────
        te = self.type_embed(x_type)                      # [N, embed_dim]
        h  = torch.cat([te, x_cont], dim=1)               # [N, embed_dim + cont_dim]
        h  = F.relu(self.input_norm(self.input_proj(h)))  # [N, hidden_dim]

        # ── GCN residual blocks + JK collection ───────────────────────────────
        xs = [h]   # collect layer outputs for JumpingKnowledge
        for block in self.gcn_blocks:
            res = h
            h   = block["conv"](h, edge_index)
            h   = F.relu(block["norm"](h))
            # FIX 5.2: dropout AFTER adding residual (was dropout(h) + res)
            h   = self.dropout(h + res)
            xs.append(h)

        h = self.jk(xs)   # element-wise max over xs → [N, hidden_dim]

        # ── GAT ───────────────────────────────────────────────────────────────
        h = F.relu(self.gat_norm(self.gat(h, edge_index)))  # [N, hidden_dim]

        # ── Graph-level pooling ───────────────────────────────────────────────
        h = self.pool(h, batch)   # [B, hidden_dim]

        # ── Classifier ────────────────────────────────────────────────────────
        h = F.relu(self.fc1(h))
        h = self.dropout(h)
        h = F.relu(self.fc2(h))
        return self.out(h)   # [B, num_classes] — raw logits


# ── Instantiate model ─────────────────────────────────────────────────────────
# FIX: removed spurious +1; len(node_vocab) already includes <UNK> at index 0.
model = DeepAstGNN(
    vocab_size     = ACTUAL_VOCAB_SIZE,
    embed_dim      = EMB_DIM,
    cont_dim       = CONT_DIM,
    hidden_dim     = HIDDEN_DIM,
    num_classes    = NUM_CLASSES,
    dropout        = 0.3,
    num_gcn_layers = NUM_GCN_LAYERS,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"DeepAstGNN (v5) — {n_params:,} parameters")
print(f"  GCN layers : {NUM_GCN_LAYERS} + JumpingKnowledge(max) + 1 GAT")
print(f"  Vocab size : {ACTUAL_VOCAB_SIZE}")

# ── Smoke test: forward pass on a real mini-batch ─────────────────────────────
_smoke_loader = PyGDataLoader(train_dataset, batch_size=4, shuffle=False)
_batch = next(iter(_smoke_loader)).to(device)
with torch.no_grad():
    _out = model(_batch.x_type, _batch.x_cont, _batch.edge_index, _batch.batch)
assert _out.shape == (4, NUM_CLASSES), f"Unexpected output shape: {_out.shape}"
print(f"\n✓ Forward pass smoke test passed. Output shape: {_out.shape}")
del _smoke_loader, _batch, _out

## Section 8: Training Pipeline

**v5 fixes:**
- `ReduceLROnPlateau` patience raised 1 → `SCHEDULER_PATIENCE` (3)
- Early-stop `PATIENCE` raised 3 → 5
- Baseline macro-F1 (majority class) printed each epoch — if val F1 == baseline,
  class collapse is immediately visible without inspecting a confusion matrix
- `va` dict initialised before the loop — no `NameError` if val loader is absent
- Best-epoch validation results saved separately (`best_val_results`) for use in
  the visualisation cell, independent of which epoch `va` currently holds

In [ ]:
# ========================== SECTION 8: TRAINING LOOP ==========================
from sklearn.utils.class_weight import compute_class_weight


def compute_class_weights_tensor(df: pd.DataFrame) -> torch.Tensor:
    """Compute balanced class weights and return as a device tensor."""
    y       = df["label"].astype(int).to_numpy()
    classes = np.arange(NUM_CLASSES)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return torch.tensor(weights, dtype=torch.float, device=device)


def _baseline_f1(labels: list) -> float:
    """Macro-F1 of a majority-class predictor — lower bound for model performance."""
    maj = max(set(labels), key=labels.count)
    return f1_score(labels, [maj] * len(labels), average="macro", zero_division=0)


def train_epoch(model, loader, optimizer, criterion, dev) -> dict:
    model.train()
    total_loss, all_p, all_t = 0.0, [], []
    for batch in tqdm(loader, desc="  Train", leave=False):
        batch = batch.to(dev)
        optimizer.zero_grad()
        out  = model(batch.x_type, batch.x_cont, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * batch.y.size(0)
        all_p.extend(out.argmax(1).cpu().numpy())
        all_t.extend(batch.y.cpu().numpy())
    n = max(len(all_t), 1)
    return {
        "loss": total_loss / n,
        "f1":   f1_score(all_t, all_p, average="macro", zero_division=0),
    }


@torch.no_grad()
def eval_epoch(model, loader, criterion, dev) -> dict:
    model.eval()
    total_loss, all_p, all_t = 0.0, [], []
    for batch in tqdm(loader, desc="  Eval ", leave=False):
        batch = batch.to(dev)
        out  = model(batch.x_type, batch.x_cont, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        total_loss += loss.item() * batch.y.size(0)
        all_p.extend(out.argmax(1).cpu().numpy())
        all_t.extend(batch.y.cpu().numpy())
    n = max(len(all_t), 1)
    return {
        "loss":    total_loss / n,
        "f1":      f1_score(all_t, all_p, average="macro", zero_division=0),
        "bal_acc": balanced_accuracy_score(all_t, all_p),
        "preds":   all_p,
        "labels":  all_t,
    }


# ── Setup ─────────────────────────────────────────────────────────────────────
class_weights = compute_class_weights_tensor(train_df)
print(f"Class weights : {class_weights.cpu().numpy()}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
# FIX 4.1: patience raised to SCHEDULER_PATIENCE=3 (was 1 — halved LR every epoch)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=SCHEDULER_PATIENCE,
    min_lr=1e-6
)

# FIX 6.4: initialise va before the loop so the visualisation cell never hits NameError
va: dict = {"loss": 0.0, "f1": 0.0, "bal_acc": 0.0, "preds": [], "labels": []}
best_val_results: dict = va.copy()   # will hold results from the epoch that set best_f1

best_f1     = 0.0
patience_ctr = 0
history = {
    "train_loss": [],
    "val_loss": [],
    "train_f1": [],
    "val_f1": [],
    "val_bal_acc": []
}

# Compute baseline F1 (majority class) once — any epoch where val_f1 == baseline
# means the model has collapsed to predicting the majority class.
_val_labels_list = val_df["label"].tolist() if val_df is not None else train_df["label"].tolist()
BASELINE_F1 = _baseline_f1(_val_labels_list)

print(f"\nMajority-class baseline F1 : {BASELINE_F1:.4f}")
print(f"Training for up to {NUM_EPOCHS} epochs (early-stop patience={PATIENCE})")
print("-" * 72)

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    tr = train_epoch(model, train_loader, optimizer, criterion, device)

    if val_loader is not None:
        va      = eval_epoch(model, val_loader, criterion, device)
        cur_f1  = va["f1"]
        history["val_loss"].append(va["loss"])
        history["val_f1"].append(va["f1"])
        history["val_bal_acc"].append(va["bal_acc"])
    else:
        cur_f1 = tr["f1"]
        history["val_loss"].append(0.0)
        history["val_f1"].append(0.0)
        history["val_bal_acc"].append(0.0)

    history["train_loss"].append(tr["loss"])
    history["train_f1"].append(tr["f1"])

    elapsed = time.time() - t0
    collapse_flag = " ⚠ COLLAPSED" if abs(cur_f1 - BASELINE_F1) < 0.01 else ""
    print(
        f"Ep {epoch:02d} | "
        f"TrLoss={tr['loss']:.4f} TrF1={tr['f1']:.4f} | "
        f"VaLoss={va['loss']:.4f} VaF1={cur_f1:.4f} | "
        f"Baseline={BASELINE_F1:.4f}{collapse_flag} | "
        f"LR={optimizer.param_groups[0]['lr']:.2e} | {elapsed:.1f}s",
        end="",
    )

    scheduler.step(cur_f1)

    with open("training_history.json", "w") as f:
        json.dump(history, f)

    if cur_f1 > best_f1:
        best_f1          = cur_f1
        patience_ctr     = 0
        best_val_results = va.copy()   # save predictions from this epoch for vis
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(" ★ saved")
    else:
        patience_ctr += 1
        print()
        if patience_ctr >= PATIENCE:
            print(f"\nEarly stop at epoch {epoch} (patience={PATIENCE}).")
            break

print(f"\n✓ Training complete.  Best val macro-F1 = {best_f1:.4f}")

## Section 9 & 10: Inference & Submission

Generates a `submission.csv` compatible with the SemEval scorer:
- Two columns: `id` (original row ID) and `label` (integer label ID)
- Labels are mapped back through `IDX2LABEL` to match the original label space

In [ ]:
# ========================== SECTION 9 & 10: INFERENCE & SUBMISSION ==========================
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device, weights_only=True))
print(f"✓ Best checkpoint loaded from {CHECKPOINT_PATH}")


@torch.no_grad()
def generate_submission(model, loader, dev: torch.device) -> pd.DataFrame:
    """Run model on loader and return a submission DataFrame."""
    model.eval()
    o_ids, labels = [], []
    for batch in tqdm(loader, desc="Inference"):
        batch = batch.to(dev)
        out   = model(batch.x_type, batch.x_cont, batch.edge_index, batch.batch)
        preds = out.argmax(1).cpu().numpy()
        labels.extend([IDX2LABEL.get(int(p), int(p)) for p in preds])
        o_ids.extend(batch.orig_id.cpu().numpy())
    return pd.DataFrame({"id": o_ids, "label": labels})


# ── Locate test parquet ───────────────────────────────────────────────────────
test_paths = [
    "/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A/test.parquet",
    "./task_a/sample_test.parquet",
]
test_path = next((p for p in test_paths if os.path.exists(p)), None)

if test_path:
    print(f"\nRunning inference on test set: {test_path}")
    test_df = pd.read_parquet(test_path)
    if "orig_id" not in test_df.columns:
        test_df["orig_id"] = test_df["id"].values if "id" in test_df.columns else range(len(test_df))
    test_dataset = ChunkedGraphDataset(
        test_df, parsers, node_vocab, CACHE_DIR, MAX_NODES,
        "task_a_test", CHUNK_SIZE, N_JOBS,
    )
    test_loader = PyGDataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    sub = generate_submission(model, test_loader, device)
else:
    print("\n⚠ Test set not found — generating submission on validation set.")
    sub = generate_submission(model, val_loader if val_loader is not None else train_loader, device)

# ── Validate submission format ────────────────────────────────────────────────
assert list(sub.columns) == ["id", "label"], "Submission must have exactly columns [id, label]"
valid_labels = set(IDX2LABEL.values())
bad = sub[~sub["label"].isin(valid_labels)]
if len(bad):
    print(f"⚠ {len(bad)} rows have invalid label values — check IDX2LABEL mapping.")
else:
    print(f"✓ Submission format valid — {len(sub):,} rows, labels: {sorted(sub['label'].unique())}")

out_path = "/kaggle/working/submission_v5.csv" if os.path.exists("/kaggle") else "./submission_v5.csv"
sub.to_csv(out_path, index=False)
print(f"✓ Saved: {out_path}")
print(sub.head())

## Section 10: Training Visualization & Evaluation

**v5 fix (4.7):** Confusion matrix and classification report are now generated by
re-evaluating the **best checkpoint** with a fresh `eval_epoch` call after loading.
In v4 the confusion matrix used `va` from the *last* training epoch (the one that
triggered early stopping), which is a different (worse) model than what was saved.

In [ ]:
# ========================== SECTION 10: TRAINING VISUALIZATION & EVALUATION ==========================
import seaborn as sns
sns.set_style("whitegrid")

epochs_range = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Loss curve ────────────────────────────────────────────────────────────────
axes[0].plot(epochs_range, history["train_loss"], label="Train Loss", marker="o")
axes[0].plot(epochs_range, history["val_loss"],   label="Val Loss",   marker="o")
axes[0].axhline(y=min(history["val_loss"]) if history["val_loss"] else 0,
                linestyle="--", color="gray", alpha=0.5, label="Best val loss")
axes[0].set_title("Training vs Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

# ── F1 curve ─────────────────────────────────────────────────────────────────
axes[1].plot(epochs_range, history["train_f1"], label="Train F1", marker="o")
axes[1].plot(epochs_range, history["val_f1"],   label="Val F1",   marker="o")
axes[1].axhline(y=BASELINE_F1, linestyle="--", color="red", alpha=0.7,
                label=f"Baseline (majority) = {BASELINE_F1:.3f}")
axes[1].axhline(y=best_f1, linestyle="--", color="green", alpha=0.7,
                label=f"Best val F1 = {best_f1:.3f}")
axes[1].set_title("Training vs Validation Macro-F1")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()

plt.tight_layout()
fig_path = "/kaggle/working/training_curves_v5.png" if os.path.exists("/kaggle") else "./training_curves_v5.png"
plt.savefig(fig_path, dpi=150)
plt.show()
print(f"✓ Saved training curves: {fig_path}")

# ── Confusion matrix using BEST checkpoint (FIX 4.7) ─────────────────────────
# Re-run eval_epoch on the best-checkpoint model, not the last-epoch `va` variable.
if val_loader is not None:
    print("\nEvaluating best checkpoint on validation set…")
    # Model is already loaded (best checkpoint loaded in Section 9)
    best_eval = eval_epoch(model, val_loader, criterion, device)

    print(f"\n{'='*50}")
    print(f"  Best checkpoint — Validation Results")
    print(f"{'='*50}")
    print(f"  Macro F1       : {best_eval['f1']:.4f}")
    print(f"  Balanced Acc   : {best_eval['bal_acc']:.4f}")
    print(f"  Majority F1    : {BASELINE_F1:.4f}  (collapse threshold)")
    print(f"  Δ above baseline: {best_eval['f1'] - BASELINE_F1:+.4f}")

    cm = confusion_matrix(best_eval["labels"], best_eval["preds"])

    fig2, ax2 = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=ax2,
        xticklabels=[IDX2LABEL.get(i, i) for i in range(NUM_CLASSES)],
        yticklabels=[IDX2LABEL.get(i, i) for i in range(NUM_CLASSES)],
    )
    ax2.set_title("Confusion Matrix — Best Checkpoint (v5)")
    ax2.set_xlabel("Predicted")
    ax2.set_ylabel("True")
    plt.tight_layout()
    cm_path = "/kaggle/working/confusion_matrix_v5.png" if os.path.exists("/kaggle") else "./confusion_matrix_v5.png"
    plt.savefig(cm_path, dpi=150)
    plt.show()
    print(f"\n✓ Saved confusion matrix: {cm_path}")

    print("\nClassification Report:")
    print(classification_report(
        best_eval["labels"], best_eval["preds"],
        target_names=[str(IDX2LABEL.get(i, i)) for i in range(NUM_CLASSES)],
        digits=4,
        zero_division=0,
    ))
else:
    print("No validation loader available — skipping confusion matrix.")

## Section 11 — Pure CodeBERT Baseline (Optional)

Commented out. Uncomment to run a fine-tuned CodeBERT sequence classifier as a
comparison baseline.  Note that CodeBERT can only be added here as a separate
experiment — per the v5 review, mixing CodeBERT embeddings into the GNN output
(as done in v3) creates architectural tension: CodeBERT's 768D surface-level
representation dominates the GNN's 128–256D structural output, causing the model
to ignore graph structure entirely.

In [ ]:
# CodeBERT baseline — uncomment to run as a separate comparison experiment.
# See review notes: do NOT combine CodeBERT embeddings with GNN output.

# class CodeBERTBaseline:
#     """Fine-tuned CodeBERT sequence classifier for binary code detection."""
#     def __init__(self, model_name="microsoft/codebert-base", max_length=512):
#         from transformers import (
#             RobertaTokenizer, RobertaForSequenceClassification,
#             TrainingArguments, Trainer, EarlyStoppingCallback
#         )
#         self.model_name = model_name
#         self.max_length = max_length
#
#     def run(self, train_df, val_df, num_labels, output_dir="./codebert_output"):
#         # tokenize, train, evaluate ...
#         pass

print("Section 11 placeholder — CodeBERT baseline is commented out.")